# 기초계산과학 Tutorial Session 초안
## 주제: CNN을 이용한 PID 성능 개선

대상: 학부 2학년, 파이썬 초심자

---
### 오늘의 큰 흐름 (한 번만 정의)
**raw detector data → preprocessing → dataset → loss → optimizer/GD → train/validation/test → CNN → evaluation → visualization**

> 왜 이 순서가 중요한가?
> - 실험 데이터는 바로 학습에 넣을 수 없음(형태/스케일/노이즈 문제)
> - 모델은 "무엇이 정답인지"(loss)와 "어떻게 고칠지"(optimizer)가 있어야 학습 가능
> - 성능 평가는 학습 데이터만으로 하면 과적합을 놓치기 쉬움


## Part 1. 문제 정의와 데이터 이해
### Learning Objective
- PID(Particle Identification) 문제를 "입력-출력" 관점으로 설명할 수 있다.
- detector raw data가 왜 전처리되어야 하는지 말할 수 있다.

### 쉬운 설명
PID는 "이 신호가 어떤 입자인가?"를 맞추는 분류 문제다.
원시 검출기 데이터(raw)는 단위, 범위, 결측, 노이즈가 섞여 있어 그대로는 학습이 불안정하다.


In [ ]:
import numpy as np
import pandas as pd

# 예시용 더미 raw 데이터 (실습 시 실제 파일로 교체)
rng = np.random.default_rng(7)
N = 200
raw = pd.DataFrame({
    "dx": rng.normal(0, 1, N),
    "dy": rng.normal(0, 1, N),
    "dz": rng.normal(0, 2.2, N),   # z축은 분해능이 낮다고 가정
    "energy": rng.uniform(0.1, 5.0, N),
    "pid_label": rng.integers(0, 3, N)  # 0: proton, 1: pion, 2: alpha (예시)
})
raw.head()


## Part 2. Preprocessing과 Dataset 만들기
### Learning Objective
- 정규화/결측치 처리의 목적을 설명할 수 있다.
- 학습 가능한 tensor dataset 형태로 바꿀 수 있다.

### 쉬운 설명
모델은 숫자 크기에 민감하다. 같은 의미의 변수라도 스케일이 다르면 한쪽만 과도하게 반영된다.
그래서 전처리로 입력을 비슷한 범위로 맞춘다.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["dx", "dy", "dz", "energy"]
X = raw[features].copy()
y = raw["pid_label"].copy()

# 간단한 결측치 처리 예시
X = X.fillna(X.median(numeric_only=True))

# train/val/test = 70/15/15
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_s.shape, X_val_s.shape, X_test_s.shape


## Part 3. Loss, Optimizer(GD), 그리고 학습 루프
### Learning Objective
- loss가 "현재 오답 정도"라는 점을 설명할 수 있다.
- gradient descent/optimizer가 loss를 줄이는 방향으로 파라미터를 업데이트함을 이해한다.

### 쉬운 설명
- **Loss**: 시험에서 몇 점 틀렸는지 숫자로 표현한 것.
- **Optimizer**: 틀린 원인을 따라가며 가중치를 조금씩 고치는 규칙.
- **GD(경사하강법)**: loss가 낮아지는 방향으로 이동.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 1D CNN 입력 형태를 위해 (batch, channel=1, length=4)
Xtr = torch.tensor(X_train_s, dtype=torch.float32).unsqueeze(1)
Xva = torch.tensor(X_val_s, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_test_s, dtype=torch.float32).unsqueeze(1)

ytr = torch.tensor(y_train.to_numpy(), dtype=torch.long)
yva = torch.tensor(y_val.to_numpy(), dtype=torch.long)
yte = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=64)

def make_model(num_classes=3):
    return nn.Sequential(
        nn.Conv1d(1, 8, kernel_size=2), nn.ReLU(),
        nn.Conv1d(8, 16, kernel_size=2), nn.ReLU(),
        nn.Flatten(),
        nn.Linear(16*2, 32), nn.ReLU(),
        nn.Linear(32, num_classes)
    )

model = make_model()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb)
        loss = criterion(logits, yb)

        if train_mode:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_n += xb.size(0)

    return total_loss/total_n, total_correct/total_n

for epoch in range(1, 6):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, None)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.3f}, acc {tr_acc:.3f} | val loss {va_loss:.3f}, acc {va_acc:.3f}")


## Part 4. Evaluation + Reasoning Example (핵심)
### Learning Objective
- 정확도 하나만 보지 않고 confusion matrix/클래스별 성능을 확인할 수 있다.
- 입력 표현의 한계를 근거를 들어 설명할 수 있다.

### 쉬운 설명
모델이 잘 맞추는 경우와 못 맞추는 경우를 분해해서 봐야 개선 포인트가 보인다.
특히 detector 좌표 표현에서 축마다 분해능 차이가 있으면 성능 불균형이 생길 수 있다.

### Reasoning 예시: 왜 `dx/dy`는 잘 맞고 `dz/dy`는 어렵나?
- `dx/dy`: x,y는 readout pitch가 촘촘하고 전하 분포가 비교적 안정적이라 shape 정보가 잘 남는다.
- `dz/dy`: z는 drift time, diffusion, 전자 수집/증폭 변동 영향이 커서 분포가 넓어지고 class overlap이 증가한다.
- 결과적으로 같은 CNN 구조라도 z축 관련 feature는 decision boundary가 흐려져 오분류가 늘 수 있다.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    pred = model(Xte).argmax(dim=1).cpu().numpy()

y_true = yte.cpu().numpy()
print(classification_report(y_true, pred, digits=3))
print("Confusion matrix:
", confusion_matrix(y_true, pred))


## Part 5. Visualization과 해석
### Learning Objective
- 학습 곡선과 feature 관계를 그림으로 설명할 수 있다.
- "성능 숫자"를 "물리적/실험적 이유"와 연결해 해석할 수 있다.

### 쉬운 설명
시각화는 결과를 검증하고, 다음 실험(특징 추가/모델 변경)의 우선순위를 정하는 도구다.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].scatter(raw["dx"], raw["dy"], c=raw["pid_label"], s=12, alpha=0.7)
ax[0].set_title("dx vs dy (class-colored)")
ax[0].set_xlabel("dx"); ax[0].set_ylabel("dy")

ax[1].scatter(raw["dz"], raw["dy"], c=raw["pid_label"], s=12, alpha=0.7)
ax[1].set_title("dz vs dy (class-colored)")
ax[1].set_xlabel("dz"); ax[1].set_ylabel("dy")

plt.tight_layout(); plt.show()


## 마무리 체크 질문 (학생 토론용)
1. 전처리를 하지 않으면 어떤 feature가 과도하게 영향력을 가지는가?
2. train acc는 높은데 val/test acc가 낮으면 무엇을 의심해야 하는가?
3. dz 관련 성능이 낮다면 데이터 표현을 어떻게 바꿔볼 수 있는가? (예: time bin 재구성, 추가 calibration feature)
4. CNN 대신 MLP/Transformer를 쓸 때 장단점은 무엇인가?
